# Assignment 3 — MM-IMDb M3

- Image-only: ResNet-50, fine-tuned.
- Text-only: BERT-base-uncased, fine-tuned.
- Simple fusion: frozen encoders, cached features, concatenation head.
- Improved fusion: Gated Multimodal Unit (GMU).
- Ablations: GMU with and without modality dropout, plus missing-modality evaluation.
- Analysis: per-class metrics, support/conflict cases, and per-class error counts.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "assignment3").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "assignment3" / "src"))

import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

from utils.mmimdb import (
    build_label_matrix,
    build_splits,
    get_genres,
    load_mmimdb,
    make_image_loader,
    make_text_loader,
)
from models import (
    TEXT_MODEL_NAME,
    build_fusion_head,
    build_gmu,
    build_image_encoder,
    build_image_model,
    build_text_encoder,
    build_text_model,
)
from training_loop import (
    extract_image_features,
    extract_text_features,
    fit_fusion,
    fit_gmu,
    fit_image,
    fit_text,
    metrics_from_probabilities,
    run_feature_epoch,
    run_gmu_epoch,
    run_image_epoch,
    run_text_epoch,
)

SEED = 42
IMAGE_EPOCHS = 10
TEXT_EPOCHS = 5
FUSION_EPOCHS = 30
GMU_EPOCHS = 30
IMAGE_BATCH = 256
TEXT_BATCH = 64
FUSION_BATCH = 256
WORKERS = 8
LIMIT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = ROOT / "assignment3" / "results"
CHECKPOINT_DIR = ROOT / "assignment3" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
print("device:", DEVICE)

device: cuda


In [2]:
dataset = load_mmimdb()
genres = get_genres(dataset)
labels = build_label_matrix(dataset, genres)
splits = build_splits(dataset, genres)

if LIMIT:
    splits = {name: indices[:LIMIT] for name, indices in splits.items()}
print("genres:", len(genres))
print("splits:", {name: len(indices) for name, indices in splits.items()})

genres: 23
splits: {'train': 15558, 'dev': 2595, 'test': 7806}


In [3]:
image_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
image_model = build_image_model(len(genres)).to(DEVICE)
image_history, image_best_epoch, image_thresholds = fit_image(
    image_model,
    image_loaders,
    DEVICE,
    epochs=IMAGE_EPOCHS,
    lr=3e-4,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "image_resnet50.pt",
)

_, image_test_probs, image_test_targets = run_image_epoch(
    image_model, image_loaders["test"], criterion, DEVICE, AMP
)
image_metrics = metrics_from_probabilities(
    image_test_probs, image_test_targets, image_thresholds
)
print({key: image_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/10 | train loss 0.3023 | val loss 0.2604 macro-F1 0.3081 | 16.7s


epoch 02/10 | train loss 0.2355 | val loss 0.2511 macro-F1 0.3414 | 13.9s


epoch 03/10 | train loss 0.2090 | val loss 0.2522 macro-F1 0.3509 | 13.8s


epoch 04/10 | train loss 0.1797 | val loss 0.2798 macro-F1 0.3438 | 13.7s


epoch 05/10 | train loss 0.1508 | val loss 0.2868 macro-F1 0.3490 | 14.1s


epoch 06/10 | train loss 0.1248 | val loss 0.2965 macro-F1 0.3431 | 14.3s


epoch 07/10 | train loss 0.1043 | val loss 0.3137 macro-F1 0.3306 | 14.1s


epoch 08/10 | train loss 0.0868 | val loss 0.3348 macro-F1 0.3352 | 13.7s


epoch 09/10 | train loss 0.0701 | val loss 0.3717 macro-F1 0.3397 | 14.8s


epoch 10/10 | train loss 0.0571 | val loss 0.3823 macro-F1 0.3332 | 13.7s


{'macro_f1': 0.33444039474269244, 'micro_f1': 0.43654161157269444, 'samples_f1': 0.4393892385467589, 'exact_match': 0.035613630540609786}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
text_model = build_text_model(len(genres)).to(DEVICE)
text_history, text_best_epoch, text_thresholds = fit_text(
    text_model,
    text_loaders,
    DEVICE,
    epochs=TEXT_EPOCHS,
    lr=2e-5,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "text_bert.pt",
)

_, text_test_probs, text_test_targets = run_text_epoch(
    text_model, text_loaders["test"], criterion, DEVICE, AMP
)
text_metrics = metrics_from_probabilities(
    text_test_probs, text_test_targets, text_thresholds
)
print({key: text_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch 01/5 | train loss 0.3250 | val loss 0.2489 macro-F1 0.4104 | 26.2s


epoch 02/5 | train loss 0.2223 | val loss 0.2089 macro-F1 0.5107 | 25.5s


epoch 03/5 | train loss 0.1881 | val loss 0.1955 macro-F1 0.5557 | 26.9s


epoch 04/5 | train loss 0.1644 | val loss 0.1877 macro-F1 0.5706 | 25.2s


epoch 05/5 | train loss 0.1441 | val loss 0.1899 macro-F1 0.5881 | 25.2s


{'macro_f1': 0.576055913510924, 'micro_f1': 0.6509765118854101, 'samples_f1': 0.6555437914776885, 'exact_match': 0.16974122469894953}


In [5]:
image_encoder = build_image_encoder().to(DEVICE)
text_encoder = build_text_encoder().to(DEVICE)

image_feature_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, False, WORKERS)
    for name, indices in splits.items()
}
text_feature_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, False, WORKERS)
    for name, indices in splits.items()
}

image_feature_map = {}
text_feature_map = {}
fusion_features = {}
fusion_labels = {}
for name in splits:
    image_features, image_targets = extract_image_features(
        image_encoder, image_feature_loaders[name], DEVICE, AMP
    )
    text_features, text_targets = extract_text_features(
        text_encoder, text_feature_loaders[name], DEVICE, AMP
    )
    assert torch.equal(image_targets, text_targets)
    image_feature_map[name] = image_features
    text_feature_map[name] = text_features
    fusion_features[name] = torch.cat([image_features, text_features], dim=1)
    fusion_labels[name] = image_targets

fusion_head = build_fusion_head(
    input_dim=fusion_features["train"].shape[1],
    hidden_dim=512,
    num_labels=len(genres),
).to(DEVICE)
fusion_history, fusion_best_epoch, fusion_thresholds = fit_fusion(
    fusion_head,
    fusion_features["train"],
    fusion_labels["train"],
    fusion_features["dev"],
    fusion_labels["dev"],
    DEVICE,
    epochs=FUSION_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=FUSION_BATCH,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "fusion_mlp.pt",
)

test_loader = DataLoader(
    TensorDataset(fusion_features["test"], fusion_labels["test"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
_, fusion_test_probs, fusion_test_targets = run_feature_epoch(
    fusion_head, test_loader, criterion, DEVICE
)
fusion_metrics = metrics_from_probabilities(
    fusion_test_probs, fusion_test_targets, fusion_thresholds
)
print({key: fusion_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/30 | train loss 0.2705 | val macro-F1 0.4488


epoch 05/30 | train loss 0.1723 | val macro-F1 0.5733


epoch 10/30 | train loss 0.1545 | val macro-F1 0.5922


epoch 15/30 | train loss 0.1434 | val macro-F1 0.5974


epoch 20/30 | train loss 0.1343 | val macro-F1 0.5976


epoch 25/30 | train loss 0.1264 | val macro-F1 0.5993


epoch 30/30 | train loss 0.1181 | val macro-F1 0.5996
{'macro_f1': 0.5868550643893883, 'micro_f1': 0.6451399081482885, 'samples_f1': 0.6417673353491954, 'exact_match': 0.14078913656161926}


In [6]:
gmu_variants = {}
for name, dropout in [("gmu_no_dropout", 0.0), ("gmu_dropout", 0.2)]:
    print(f"\n=== {name} ===")
    gmu = build_gmu(num_labels=len(genres), hidden_dim=512, modality_dropout=dropout).to(DEVICE)
    history, best_epoch, thresholds = fit_gmu(
        gmu,
        image_feature_map["train"],
        text_feature_map["train"],
        fusion_labels["train"],
        image_feature_map["dev"],
        text_feature_map["dev"],
        fusion_labels["dev"],
        DEVICE,
        epochs=GMU_EPOCHS,
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=FUSION_BATCH,
        modality_dropout=dropout,
        seed=SEED,
        checkpoint_path=CHECKPOINT_DIR / f"{name}.pt",
    )
    gmu_variants[name] = {
        "model": gmu,
        "history": history,
        "best_epoch": best_epoch,
        "thresholds": thresholds,
    }


=== gmu_no_dropout ===


epoch 01/30 | train loss 0.2708 | val macro-F1 0.4234


epoch 05/30 | train loss 0.1705 | val macro-F1 0.5807


epoch 10/30 | train loss 0.1538 | val macro-F1 0.5899


epoch 15/30 | train loss 0.1431 | val macro-F1 0.5976


epoch 20/30 | train loss 0.1343 | val macro-F1 0.5989


... (5 outputs left out)

epoch 15/30 | train loss 0.1726 | val macro-F1 0.6019


epoch 20/30 | train loss 0.1665 | val macro-F1 0.6050


epoch 25/30 | train loss 0.1635 | val macro-F1 0.5995


epoch 30/30 | train loss 0.1601 | val macro-F1 0.5982


In [7]:
gmu_metrics = {}
gmu_probabilities = {}
gmu_missing_metrics = {}

for name, variant in gmu_variants.items():
    gmu = variant["model"]
    thresholds = variant["thresholds"]
    test_loader = DataLoader(
        TensorDataset(
            image_feature_map["test"],
            text_feature_map["test"],
            fusion_labels["test"],
        ),
        batch_size=FUSION_BATCH,
        shuffle=False,
    )
    _, probabilities, targets = run_gmu_epoch(gmu, test_loader, criterion, DEVICE, AMP)
    gmu_metrics[name] = metrics_from_probabilities(probabilities, targets, thresholds)
    gmu_probabilities[name] = probabilities

    missing = {}
    for drop_mode in ("image", "text"):
        _, dropped_probs, dropped_targets = run_gmu_epoch(
            gmu, test_loader, criterion, DEVICE, AMP, drop_mode=drop_mode
        )
        missing[drop_mode] = metrics_from_probabilities(
            dropped_probs, dropped_targets, thresholds
        )
    gmu_missing_metrics[name] = missing

for name, metrics in gmu_metrics.items():
    print(name, {key: metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})
for name, missing in gmu_missing_metrics.items():
    print(name, "image dropped:", missing["image"]["macro_f1"], "text dropped:", missing["text"]["macro_f1"])


gmu_no_dropout {'macro_f1': 0.5847415156069673, 'micro_f1': 0.6466082627220068, 'samples_f1': 0.644217418929179, 'exact_match': 0.14885985139636176}
gmu_dropout {'macro_f1': 0.588670702938291, 'micro_f1': 0.6481499187302802, 'samples_f1': 0.6450302977974001, 'exact_match': 0.15078145016653857}
gmu_no_dropout image dropped: 0.530666645964423 text dropped: 0.27890094459264714
gmu_dropout image dropped: 0.5574152976569354 text dropped: 0.2904328385186324


In [8]:
def case_stats(predictions_a, predictions_b, targets):
    correct_a = (predictions_a == targets).all(axis=1)
    correct_b = (predictions_b == targets).all(axis=1)
    return {
        "both_correct": int(np.sum(correct_a & correct_b)),
        "a_only": int(np.sum(correct_a & ~correct_b)),
        "b_only": int(np.sum(~correct_a & correct_b)),
        "both_wrong": int(np.sum(~correct_a & ~correct_b)),
    }


def per_class_error_counts(predictions, targets):
    false_positive = ((predictions == 1) & (targets == 0)).sum(axis=0)
    false_negative = ((predictions == 0) & (targets == 1)).sum(axis=0)
    return false_positive, false_negative


all_probabilities = {
    "image_only": image_test_probs,
    "text_only": text_test_probs,
    "simple_fusion": fusion_test_probs,
    **gmu_probabilities,
}
all_thresholds = {
    "image_only": image_thresholds,
    "text_only": text_thresholds,
    "simple_fusion": fusion_thresholds,
    **{name: variant["thresholds"] for name, variant in gmu_variants.items()},
}
all_predictions = {
    name: (probabilities >= all_thresholds[name][None, :]).astype(np.int32)
    for name, probabilities in all_probabilities.items()
}
all_metrics = {
    "image_only": image_metrics,
    "text_only": text_metrics,
    "simple_fusion": fusion_metrics,
    **gmu_metrics,
}

support_analysis = {}
for name in ("simple_fusion", "gmu_dropout", "gmu_no_dropout"):
    support_analysis[name] = case_stats(
        all_predictions[name],
        all_predictions["text_only"],
        fusion_labels["test"].numpy(),
    )

error_counts = {}
for name, predictions in all_predictions.items():
    fp, fn = per_class_error_counts(predictions, fusion_labels["test"].numpy())
    ranked = sorted(
        [
            {
                "genre": genres[index],
                "false_positive": int(fp[index]),
                "false_negative": int(fn[index]),
                "total": int(fp[index] + fn[index]),
            }
            for index in range(len(genres))
        ],
        key=lambda item: item["total"],
        reverse=True,
    )
    error_counts[name] = ranked[:10]

example_cases = {}
for name in ("simple_fusion", "gmu_dropout"):
    a = all_predictions[name]
    b = all_predictions["text_only"]
    correct_a = (a == fusion_labels["test"].numpy()).all(axis=1)
    correct_b = (b == fusion_labels["test"].numpy()).all(axis=1)
    masks = {
        "fusion_correct_text_wrong": correct_a & ~correct_b,
        "fusion_wrong_text_correct": ~correct_a & correct_b,
        "both_wrong": ~correct_a & ~correct_b,
        "both_correct": correct_a & correct_b,
    }
    example_cases[name] = {
        case: np.flatnonzero(mask)[:20].tolist()
        for case, mask in masks.items()
    }

np.savez(
    RESULTS_DIR / "m3_predictions.npz",
    targets=fusion_labels["test"].numpy(),
    **{f"pred_{name}": pred for name, pred in all_predictions.items()},
)
print("support analysis:", support_analysis)

support analysis: {'simple_fusion': {'both_correct': 528, 'a_only': 571, 'b_only': 797, 'both_wrong': 5910}, 'gmu_dropout': {'both_correct': 576, 'a_only': 601, 'b_only': 749, 'both_wrong': 5880}, 'gmu_no_dropout': {'both_correct': 561, 'a_only': 601, 'b_only': 764, 'both_wrong': 5880}}


In [9]:
model_rows = []
for name, model, metrics, history, best_epoch, thresholds in [
    ("Image-only ResNet-50", image_model, image_metrics, image_history, image_best_epoch, image_thresholds),
    ("Text-only BERT", text_model, text_metrics, text_history, text_best_epoch, text_thresholds),
    ("Simple fusion", fusion_head, fusion_metrics, fusion_history, fusion_best_epoch, fusion_thresholds),
] + [
    (name, variant["model"], gmu_metrics[name], variant["history"], variant["best_epoch"], variant["thresholds"])
    for name, variant in gmu_variants.items()
]:
    model_rows.append(
        {
            "model": name,
            "params": sum(p.numel() for p in model.parameters()),
            "best_epoch": int(best_epoch),
            "train_time_s": float(sum(history["epoch_time"])),
            "test_macro_f1": metrics["macro_f1"],
            "test_micro_f1": metrics["micro_f1"],
            "test_samples_f1": metrics["samples_f1"],
            "test_exact_match": metrics["exact_match"],
        }
    )

results_df = pd.DataFrame(model_rows)
results_df

,model,params,best_epoch,train_time_s,test_macro_f1,test_micro_f1,test_samples_f1,test_exact_match
0,Image-only ResNet-50,23555159,3,142.871861,0.334440,0.436542,0.439389,0.035614
1,Text-only BERT,109499927,5,128.949250,0.576056,0.650977,0.655544,0.169741
2,Simple fusion,1454103,27,26.344540,0.586855,0.645140,0.641767,0.140789
3,gmu_no_dropout,2896919,19,28.946999,0.584742,0.646608,0.644217,0.148860
4,gmu_dropout,2896919,20,28.846662,0.588671,0.648150,0.645030,0.150781


In [10]:
config = {
    "dataset": "MM-IMDb",
    "seed": SEED,
    "image_epochs": IMAGE_EPOCHS,
    "text_epochs": TEXT_EPOCHS,
    "fusion_epochs": FUSION_EPOCHS,
    "gmu_epochs": GMU_EPOCHS,
    "image_batch": IMAGE_BATCH,
    "text_batch": TEXT_BATCH,
    "fusion_batch": FUSION_BATCH,
    "limit": LIMIT,
    "amp": AMP,
    "device": str(DEVICE),
}
payload = {
    "config": config,
    "genres": genres,
    "splits": {name: len(indices) for name, indices in splits.items()},
    "results": model_rows,
    "gmu_missing_metrics": gmu_missing_metrics,
    "support_analysis": support_analysis,
    "error_counts": error_counts,
    "example_cases": example_cases,
    "per_class_f1": {name: metrics["per_class_f1"] for name, metrics in all_metrics.items()},
    "thresholds": {
        "image": image_thresholds.tolist(),
        "text": text_thresholds.tolist(),
        "simple_fusion": fusion_thresholds.tolist(),
        **{name: variant["thresholds"].tolist() for name, variant in gmu_variants.items()},
    },
    "history": {
        "image": image_history,
        "text": text_history,
        "simple_fusion": fusion_history,
        **{name: variant["history"] for name, variant in gmu_variants.items()},
    },
}
with open(RESULTS_DIR / "m3_results.json", "w") as handle:
    json.dump(payload, handle, indent=2)
results_df.to_csv(RESULTS_DIR / "m3_results.csv", index=False)
with open(RESULTS_DIR / "m3_error_analysis.json", "w") as handle:
    json.dump(
        {
            "support_analysis": support_analysis,
            "error_counts": error_counts,
            "example_cases": example_cases,
        },
        handle,
        indent=2,
    )
print("saved", RESULTS_DIR / "m3_results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment3/results/m3_results.json
